In [1]:
import pandas as pd


In [2]:
task3 = "https://github.com/PleschNA/testGA/raw/main/task3.xlsx"
df = pd.read_excel(task3)


In [3]:
# Переименуем колонки для удобства
df.columns = ["Бренд", "Неделя", "Выручка", "Маржинальность"]

# Приведём выручку в млн руб для удобства
df["Выручка_млн"] = df["Выручка"] / 1_000_000
df["Маржинальность_%"] = df["Маржинальность"] * 100

print(df.head())
print()
print("Недели:", df["Неделя"].unique())
print("Брендов:", df["Бренд"].nunique())
print("Строк:", len(df))

      Бренд  Неделя       Выручка  Маржинальность  Выручка_млн  \
0  Shiseido      14  1.353686e+09        0.167824  1353.685521   
1   Clarins      14  1.329934e+09        0.091348  1329.934372   
2  Shiseido      15  1.267702e+09        0.169212  1267.702458   
3   Clarins      15  9.747119e+08        0.145904   974.711948   
4  Clinique      15  4.982012e+08        0.140136   498.201173   

   Маржинальность_%  
0         16.782360  
1          9.134757  
2         16.921186  
3         14.590366  
4         14.013613  

Недели: [14 15]
Брендов: 55
Строк: 104


In [4]:
# Разделим на две недели
cur = df[df["Неделя"] == 15].set_index("Бренд")
prev = df[df["Неделя"] == 14].set_index("Бренд")

# Соединим
cmp = prev.add_suffix("_пред").join(cur.add_suffix("_тек"), how="outer").fillna(0)
cmp["Δ_выручка"] = cmp["Выручка_млн_тек"] - cmp["Выручка_млн_пред"]
cmp["Δ_выручка_%"] = (cmp["Выручка_млн_тек"] - cmp["Выручка_млн_пред"]) / cmp["Выручка_млн_пред"] * 100
cmp["Δ_маржинальность_пп"] = cmp["Маржинальность_%_тек"] - cmp["Маржинальность_%_пред"]

# Общий итог
total_prev = cmp["Выручка_млн_пред"].sum()
total_cur = cmp["Выручка_млн_тек"].sum()
total_marg_prev = (cmp["Выручка_млн_пред"] * cmp["Маржинальность_%_пред"]).sum() / total_prev
total_marg_cur = (cmp["Выручка_млн_тек"] * cmp["Маржинальность_%_тек"]).sum() / total_cur

print(f"=== ИТОГИ ===")
print(f"Выручка: {total_prev:,.1f} → {total_cur:,.1f} млн руб "
      f"({(total_cur/total_prev-1)*100:+.1f}%)")
print(f"Средневзвешенная маржинальность: {total_marg_prev:.1f}% → {total_marg_cur:.1f}% "
      f"({total_marg_cur-total_marg_prev:+.1f} пп)")

=== ИТОГИ ===
Выручка: 8,464.2 → 8,207.9 млн руб (-3.0%)
Средневзвешенная маржинальность: 14.6% → 15.5% (+0.8 пп)


In [6]:
# Сортируем по изменению
top_drop = cmp.sort_values("Δ_выручка_%").head(10)[
    ["Выручка_млн_пред", "Выручка_млн_тек", "Δ_выручка_%", 
     "Маржинальность_%_пред", "Маржинальность_%_тек", "Δ_маржинальность_пп"]
].round(1)

top_grow = cmp.sort_values("Δ_выручка_%", ascending=False).head(10)[
    ["Выручка_млн_пред", "Выручка_млн_тек", "Δ_выручка_%",
     "Маржинальность_%_пред", "Маржинальность_%_тек", "Δ_маржинальность_пп"]
].round(1)

print("=== ТОП-10 по падению выручки ===")
print(top_drop.to_string())
print()
print("=== ТОП-10 по росту выручки ===")
print(top_grow.to_string())

=== ТОП-10 по падению выручки ===
                   Выручка_млн_пред  Выручка_млн_тек  Δ_выручка_%  Маржинальность_%_пред  Маржинальность_%_тек  Δ_маржинальность_пп
Бренд                                                                                                                              
Bellefontaine                  17.8              0.0       -100.0                   10.8                   0.0                -10.8
Dr Brandt                      20.1              0.0       -100.0                   13.6                   0.0                -13.6
Grown Alchemist                15.3              0.0       -100.0                   20.4                   0.0                -20.4
Anne Semonin                   32.7             19.2        -41.1                   20.2                  20.2                 -0.1
Dr Barbara Sturm               41.7             27.4        -34.4                   22.3                  19.5                 -2.9
Sisley                        301.4       

In [7]:
# Вклад в изменение (в млн руб)
cmp["Вклад_в_Δ_выручка"] = cmp["Δ_выручка"]

top_impact = cmp.reindex(cmp["Вклад_в_Δ_выручка"].abs().sort_values(ascending=False).index).head(15)
print("Топ-15 по абсолютному вкладу в изменение выручки (млн руб):")
print(top_impact[["Выручка_млн_пред", "Выручка_млн_тек", "Вклад_в_Δ_выручка", "Δ_маржинальность_пп"]].round(1).to_string())

Топ-15 по абсолютному вкладу в изменение выручки (млн руб):
             Выручка_млн_пред  Выручка_млн_тек  Вклад_в_Δ_выручка  Δ_маржинальность_пп
Бренд                                                                                 
Clarins                1329.9            974.7             -355.2                  5.5
Payot                   265.8            368.1              102.3                 -0.7
Sisley                  301.4            206.7              -94.7                  1.2
Shiseido               1353.7           1267.7              -86.0                  0.1
Valmont                 195.1            270.7               75.7                  1.0
La Mer                  228.0            274.0               46.0                  0.1
Caudalie                175.7            220.1               44.4                 -2.3
Lancaster                79.9            124.2               44.3                 -4.5
Lancome                 241.3            204.3              -37.1     